# 04_02_Loading `Fact_Punctuality` to SQL Server

In [1]:
import os
from pathlib import Path

import pandas as pd

import infrabel_punctuality as ip

In [2]:
PATH_PROCESSED = Path("../data/processed/")

# Table of Contents  

- [2. LOADING FACT TABLE](#2-loading-fact-table)

- [2.1. Preparing SQL Server Connection](#21-preparing-sql-server-connection)  

- [2.2. Loading DataFrame from `processed` Directory](#22-loading-dataframe-from-processed-directory)  

- [2.3. Loading Fact_Punctuality to SQL Server](#23-loading-fact_punctuality-to-sql-server)  

## 2. LOADING FACT TABLE

**LOADING STRATEGY**  

**This notebook loads the `Fact_Punctuality` DataFrame into the SQL Server data warehouse.**  

1) The loading of the dimension tables and the fact table has been separated into two notebooks. Although both notebooks implement the same loading strategy, the fact table volume - 45.5 million rows across 26 columns - requires significantly more execution time and memory than the dimension tables. Separating these loading processes improves readability and allows each to be executed independently.

2) To avoid implicit type conversions between the DataFrame dtypes and the SQL column types, and to ensure better control over column typing, **the fact table has already been created using an SQL DDL script**: *04_create_fact_punctuality* — located in `/sql/ddl/`.  

3) The loading strategy is a **full load**. Before each load run, a **TRUNCATE TABLE** statement is executed on the target table to prevent duplicates while preserving the DDL-defined fact table structure defined. The loading cell has been commented out after the initial load to prevent accidental re-execution.   

4) The loading process is performed through **SQLAlchemy** using the functions provided by the `sql_server_connection` module of the project's `infrabel_punctuality` package.  

5) The SQLAlchemy engine is configured to connect to the project's SQL Server database, `infrabel_punctuality_dwh`, created in *01_create_database* located in `/sql/ddl/`. The **database name** is therefore hardcoded in this notebook.  

6) The engine is also configured to connect to a local SQL Server instance using `localhost` as the default **server name**. If the SQL Server instance on your machine is not accessible through `localhost`, set the `SQL_SERVER` environment variable to the correct server or instance name.

## 2.1. Preparing SQL Server Connection

In this section:  

* The SQLAlchemy engine is created with the required parameters - driver, server and database names - and its connection to the database is tested.  

* The `select_sql_driver()` and `get_engine()` functions support both *ODBC Driver 17 for SQL Server* and *ODBC Driver 18 for SQL Server*.  

In [3]:
driver = ip.select_sql_driver()
server = os.getenv("SQL_SERVER", "localhost")
database = "infrabel_punctuality_dwh"

Driver: ODBC Driver 18 for SQL Server


In [4]:
engine = ip.get_engine(driver, server, database)

In [5]:
ip.test_connection(engine)

Connection successful to database: infrabel_punctuality_dwh


## 2.2. Loading DataFrame from `processed` Directory

In [6]:
Fact_Punctuality = pd.read_parquet(PATH_PROCESSED / "Fact_Punctuality.parquet")

## 2.3. Loading `Fact_Punctuality` to SQL Server

⚠️ **Warning**: The loading cell below is commented out to prevent the fact table from being reloaded accidentally. The full load of `Fact_Punctuality` requires significant execution time (~30 minutes). Uncomment the cell only if you intentionally want to load the table. 

In [ ]:
#ip.full_load_large_to_sql_server(
#    engine, 
#    dataframe=Fact_Punctuality, 
#    table_name="Fact_Punctuality", 
#    schema="fact"
#    )

Loading chunks:  10%|█         | 1/10 [03:09<28:26, 189.66s/it]

Chunk : 1/10 - 5,000,000 rows
Time  : 189.54 s
Speed : 26,379 rows/s
        94.97 million rows/hour


Loading chunks:  20%|██        | 2/10 [06:05<24:12, 181.51s/it]

Chunk : 2/10 - 5,000,000 rows
Time  : 175.74 s
Speed : 28,451 rows/s
        102.42 million rows/hour


Loading chunks:  30%|███       | 3/10 [08:57<20:40, 177.23s/it]

...

Chunk : 8/10 - 5,000,000 rows
Time  : 171.75 s
Speed : 29,112 rows/s
        104.80 million rows/hour


Loading chunks:  90%|█████████ | 9/10 [26:31<02:54, 174.84s/it]

Chunk : 9/10 - 5,000,000 rows
Time  : 170.42 s
Speed : 29,339 rows/s
        105.62 million rows/hour


Loading chunks: 100%|██████████| 10/10 [26:50<00:00, 161.01s/it]

Chunk : 10/10 - 541,993 rows
Time  : 18.52 s
Speed : 29,271 rows/s
        105.38 million rows/hour


In [8]:
del Fact_Punctuality